In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../../").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
import pandas as pd
import re
from tqdm.notebook import tqdm

from lib.llm.llm_client import (
    MODEL,
    MODEL_N,
    timed_completion,
    strip_thinking,
)

In [4]:
df = pd.read_csv("../../datasets/esgenius_perturbed.csv")

print(df.shape)
df.head()

(21584, 14)


,query_id,new_id,query,answer,A,B,C,D,Z,ref_page,ref_doc,source_text,perturbation_type,perturbation_level
0,831,1,"According to the IPCC AR6 Synthesis Report, wh...",A,Implementing targeted management strategies fo...,Failing to rebuild overexploited fisheries whi...,Limiting global warming but neglecting land re...,Prioritizing disaster risk management and earl...,Not sure,46,IPCC AR6 SYR.pdf,30 Summary for Policymakers Summary for Policy...,none,0
1,889,2,According to Climate Change 2023 — Synthesis R...,B,The geographical patterns of climatic impacts ...,Geographical patterns of climatic impacts for ...,The timing of reaching a specific GWL determin...,Global warming levels are defined exclusively ...,Not sure,80-81,IPCC AR6 SYR.pdf,64 Section 2 Section 1Section 2Global Warming ...,none,0
2,892,3,Which statement accurately reflects the relati...,C,High or very high confidence in attribution is...,The text explicitly states that medium confide...,Attribution of observed physical climate chang...,Increases in heavy precipitation and sea level...,Not sure,23,IPCC AR6 SYR.pdf,7 Summary for PolicymakersSummary for Policyma...,none,0
3,983,4,According to the Climate Change 2023 — Synthes...,D,Europe,Small Islands,North America,Asia,Not sure,92,IPCC AR6 SYR.pdf,76 Section 3 Section 1Section 3011.5234 011.52...,none,0
4,1077,5,According to the Climate Change 2023 — Synthes...,A,"Early warning systems, flood-proofing of build...","Afforestation, reforestation, and levees.","Agroecological practices, disaster risk manage...","Heat Health Action Plans, vector-borne disease...",Not sure,72,IPCC AR6 SYR.pdf,"56 Section 2 Section 1Section 2wetlands, range...",none,0


In [5]:
PROMPT_TEMPLATE = """
You are answering a multiple-choice question.

Use ONLY the provided source text.

Question:
{query}

Options:
A. {A}

B. {B}

C. {C}

D. {D}

Z. {Z}

Source Text:
{source_text}

Return ONLY one letter:
A
B
C
D
or Z
"""

In [6]:
VALID_ANSWERS = {"A", "B", "C", "D", "Z"}

def predict_answer(row):
    
    prompt = PROMPT_TEMPLATE.format(
        query=row["query"],
        A=row["A"],
        B=row["B"],
        C=row["C"],
        D=row["D"],
        Z=row["Z"],
        source_text=row["source_text"]
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    try:
        response, latency_ms = timed_completion(
            messages=messages,
            model=MODEL,
            n=MODEL_N,
            temperature=0,
            extra_body={
                "chat_template_kwargs": {
                "enable_thinking": False
            }
            }
        )

        text = strip_thinking(
            response.choices[0].message.content
        ).strip()

        match = re.search(r"\b([ABCDZ])\b", text)

        if match:
            return match.group(1), latency_ms

        return "INVALID"

    except Exception as e:
        print(e)
        return "ERROR"

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd

def predict_row(row):
    guess, latency = predict_answer(row)
    return {
        "query_id": row["query_id"],
        "perturbation_type": row["perturbation_type"],
        "perturbation_level": row["perturbation_level"],
        "ground_truth": row["answer"],
        "llm_guess": guess,
        "llm_latency": latency
    }

MAX_WORKERS = 8  # tune to your API rate limit

rows = [row for _, row in df.iterrows()]

predictions = [None] * len(rows)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(predict_row, row): i for i, row in enumerate(rows)}

    with tqdm(total=len(rows)) as pbar:
        for future in as_completed(futures):
            i = futures[future]
            predictions[i] = future.result()
            pbar.update(1)

results_df = pd.DataFrame(predictions)
results_df.head()

100%|██████████| 21584/21584 [54:53<00:00,  6.55it/s] 


,query_id,perturbation_type,perturbation_level,ground_truth,llm_guess,llm_latency
0,831,none,0,A,C,2038.963446
1,889,none,0,B,B,2043.324232
2,892,none,0,C,B,2040.268525
3,983,none,0,D,D,2036.492230
4,1077,none,0,A,A,2047.582230


In [8]:
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path("../../datasets/experiments/llm/classification")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_FILE = OUTPUT_DIR / f"llm_predictions_{timestamp}.csv"

results_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved to: {OUTPUT_FILE.resolve()}")

Saved to: /Users/anton/Documents/engd/courses/llmb/llmb-project-group-1/datasets/experiments/llm/classification/llm_predictions_20260624_103748.csv
